In [29]:
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
load_dotenv()
client = OpenAI()

In [4]:
sentence_1 = "Forxiga caused UTI in 8.4% of patients in DECLARE-TIMI 58"
resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentence_1
)

vector = resp.data[0].embedding
# vector

In [5]:
def embed(sentence, model="text-embedding-3-small"):
    resp = client.embeddings.create(
        model=model,
        input=sentence
    )
    return resp.data[0].embedding

v1 = embed("Forxiga caused UTI in 8.4% of patients") 
v2 = embed("Dapagliflozin leads to urinary infections")

In [7]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product / (norm_a * norm_b)

similarity = cosine_similarity(v1, v2)
print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.5360


In [8]:
Q = [0.9, 0.8, 0.7] # user questions: 
A = [0.8, 0.7, 0.6]
B = [0.7, 0.6, 0.5]

cosine_similarities = {
    "Q_A": cosine_similarity(Q, A),
    "Q_B": cosine_similarity(Q, B),
}

cosine_similarities


{'Q_A': np.float64(0.9998962099324299), 'Q_B': np.float64(0.9994375175330728)}

In [19]:
forxiga_chunks = [
    "Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.",
    "Forxiga reduces HbA1c by 0.89% vs placebo (p<0.001) in DECLARE-TIMI 58.",
    "Forxiga is contraindicated in patients with eGFR below 45 mL/min.",
    "AstraZeneca Q3 revenue grew 12% year over year.",  # irrelevant chunk
]

chunk_embeddings = [embed(chunk) for chunk in forxiga_chunks]
forxiga_vectors = np.array(chunk_embeddings, dtype="float32")

import faiss
dim = 1536
M = 32 # number of neighbors
index = faiss.IndexHNSWFlat(dim, M)
index.hnsw.ef_search = 64
index.add(forxiga_vectors)


In [20]:
index.ntotal

4

In [21]:
query = "AstraZeneca revenue?"
q_vec = np.array([embed(query)], dtype="float32") 

D,I = index.search(q_vec, k=2)

for i in I[0]:
    print(forxiga_chunks[i])

AstraZeneca Q3 revenue grew 12% year over year.
Forxiga reduces HbA1c by 0.89% vs placebo (p<0.001) in DECLARE-TIMI 58.


In [24]:
import chromadb

client = chromadb.Client()
col = client.create_collection("forxigadb")


In [ ]:

forxiga_chunks = [
    "Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.",
    "Forxiga reduces HbA1c by 0.89% vs placebo in DECLARE-TIMI 58 trial.",
    "Forxiga is contraindicated in patients with eGFR below 45 mL/min.",
    "Forxiga dose is 10mg once daily with or without food.",
    "AstraZeneca Q3 revenue grew 12% year over year.",   # irrelevant chunk
]

col.add(documents=forxiga_chunks,
ids=[str(i) for i in range(len(forxiga_chunks))]) #all-MiniLM-L6-v2


In [34]:
results = col.query(
    query_texts=["UTI percentage Forxiga"],
    n_results=2
)
results

{'ids': [['0', '2']],
 'embeddings': None,
 'documents': [['Forxiga caused UTI in 8.4% of patients vs 5.2% in placebo group.',
   'Forxiga is contraindicated in patients with eGFR below 45 mL/min.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.40674257278442383, 1.050191044807434]]}

In [ ]:
# Assignments
# Explore chromaDB